# Image Classifier Model

## Description

This is a learning project designed with the general aim of learning basic classification and ML development. The backend part of the project will be developped for serving, using ___pytorch___, and supplied to a frontend using ___fastapi___. This file contains the snadbox practice version developped before moving to the actual server files. 

Might implement some regression testing and reinforcement learning towards the end of the project.

## Data Preprocessing

This is the method to preprocess data. I will basically get an image passed to me and concert it into a tensor of the dimension 224x224 for the image in 1 dimension as it is only one image, with total dimesions at the end being a [1, 3, 224, 224] standard stuff.

In [8]:
from torchvision import transforms
import torch
from PIL import Image
from io import BytesIO

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),              # PIL Image → tensor, scales [0,255] to [0.0, 1.0]
    transforms.Normalize(               # optional, standard ImageNet stats
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

def PreProcess(image):
    pil_im = Image.open(BytesIO(image)).convert('RGB')
    tens_im = transform(pil_im)

    if (tens_im.shape != torch.Size([3, 224, 224])):
        return None

    final_im = tens_im.unsqueeze(0)

    if (final_im.shape != torch.Size([1, 3, 224, 224])):
        return None

    return final_im
    

In [9]:
# simulate the upload
with open("/Users/francoisleutou/Desktop/CodingSpace/image-classifier/backend/testimages/reminder.jpg", "rb") as f:  # rb = read bytes
    contents = f.read() 

    res = PreProcess(contents)

    print(res)

tensor([[[[-1.7412, -1.7412, -1.7412,  ..., -1.7412, -1.7412, -1.7412],
          [-1.7412, -1.7412, -1.7412,  ..., -1.7412, -1.7412, -1.7412],
          [-1.7412, -1.7412, -1.7412,  ..., -1.7412, -1.7412, -1.7412],
          ...,
          [-1.7412, -1.7412, -1.7412,  ..., -1.7412, -1.7412, -1.7412],
          [-1.7412, -1.7412, -1.7412,  ..., -1.7412, -1.7412, -1.7412],
          [-1.7412, -1.7412, -1.7412,  ..., -1.7412, -1.7412, -1.7412]],

         [[-1.4755, -1.4755, -1.4755,  ..., -1.4755, -1.4755, -1.4755],
          [-1.4755, -1.4755, -1.4755,  ..., -1.4755, -1.4755, -1.4755],
          [-1.4755, -1.4755, -1.4755,  ..., -1.4755, -1.4755, -1.4755],
          ...,
          [-1.4755, -1.4755, -1.4755,  ..., -1.4755, -1.4755, -1.4755],
          [-1.4755, -1.4755, -1.4755,  ..., -1.4755, -1.4755, -1.4755],
          [-1.4755, -1.4755, -1.4755,  ..., -1.4755, -1.4755, -1.4755]],

         [[-1.0724, -1.0724, -1.0724,  ..., -1.0724, -1.0724, -1.0724],
          [-1.0724, -1.0724, -

## Prediction

Here we have the standard code used in predicting an output that returns the top 3 predictions and their probabilities

In [29]:
from torchvision import models
from torchvision.models import ResNet50_Weights
import json
import requests

def predictions(im):
    ## Loading the model. Here we're using a resnet50 which is a good general purpose model
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.eval() ## We set our model to eval mode 
    
    ## Loading the imageNet labels
    LABELS_URL = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json" 
    class_idx = requests.get(LABELS_URL).json()
    labels = [class_idx[str(k)][1] for k in range(len(class_idx))]
    
    ## Run Inference with gradient calculation disabled for optimization
    with torch.no_grad():
        output = model(im)

    ## Get predictions
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top_3_prob, top_3_pred = torch.topk(probabilities, 3)

    res = []

    ## get the results into a dictionary
    for i in range(3):
        res.append({"label": labels[top_3_pred[i]], "probability": round(top_3_prob[i].item(), 4)})

    return res

In [30]:
with open("/Users/francoisleutou/Desktop/CodingSpace/image-classifier/backend/testimages/cyat.jpeg", "rb") as f:  # rb = read bytes
    contents = f.read() 

    res = PreProcess(contents)
    pred = predictions(res)

    for p in pred:
        print(p)


{'label': 'tiger_cat', 'probability': 0.8093}
{'label': 'tabby', 'probability': 0.125}
{'label': 'Egyptian_cat', 'probability': 0.0221}
